In [1]:
from IPython.display import display, Markdown, HTML, clear_output
from docx import Document
from docx.shared import Cm, Pt, Length, RGBColor
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT, WD_BREAK
from docx.enum.table import WD_ALIGN_VERTICAL, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement, parse_xml
from docx.oxml.ns import nsdecls
import docx.oxml.shared
import pandas as pd
import json
import re
from Bio import Entrez
import xmltodict
import urllib.request
from urllib.request import urlopen
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
import os
import numpy as np
import openpyxl
import time
import itertools
import urllib.parse  
from decimal import Decimal
import warnings
warnings.filterwarnings('ignore')

%run Redmine_Access.ipynb
%run MyCancerGenome_parsing_final.ipynb
%run ACMG.ipynb

In [2]:
patient_info=getSampleInformationFromRedmine(165)

In [3]:
# Load JSON data
with open('GS10023149158 Mutation Report_05042024-063206.json', 'r') as f:
    data = json.load(f)

# Create dataframes for other sections, handling if they don't exist
result_summary_data = json.loads(data.get('Result_Summary', "[]"))
if result_summary_data:
    result_summary_df = pd.DataFrame(result_summary_data)
else:
    result_summary_df = None

snv_data = json.loads(data.get('SNV_Identifiers', "[]"))
if snv_data:
    snv_df = pd.DataFrame(snv_data)
else:
    snv_df = None

fusion_data = json.loads(data.get('Fusion_Identifiers', "[]"))
if fusion_data:
    fusion_df = pd.DataFrame(fusion_data)
else:
    fusion_df = None

cnv_data = json.loads(data.get('CNV_Identifiers', "[]"))
if cnv_data:
    cnv_df = pd.DataFrame(cnv_data)
else:
    cnv_df = None

hrr_data = json.loads(data.get('HRR_Details', "[]"))
if hrr_data:
    hrr_df = pd.DataFrame(hrr_data)
else:
    hrr_df = None

relevant_biomarkers_data = json.loads(data.get('Relevant_Biomarkers', "[]"))
if relevant_biomarkers_data:
    biomarkers_df = pd.DataFrame(relevant_biomarkers_data)
else:
    biomarkers_df = None

In [4]:
def add_header_footer(document):
    for section in document.sections:
        # Set header
        header = section.header
        header_paragraph = header.paragraphs[0] if header.paragraphs else header.add_paragraph()
        header_paragraph.paragraph_format.left_indent = Cm(2.7) 
        image_path = os.path.join("Heading_Images", "G-KnowMe_Logo.png")
        header_run = header_paragraph.add_run()
        header_run.add_picture(image_path, width=Cm(5.4), height=Cm(2.71))
        
        # Set footer
        footer = section.footer
        footer_paragraph = footer.paragraphs[0] if footer.paragraphs else footer.add_paragraph()
        footer_paragraph.paragraph_format.left_indent = Cm(2.7)  
        footer_run = footer_paragraph.add_run("Private and confidential")
        footer_run.font.size = Pt(12)
        footer_run.font.name = 'Times New Roman'
        footer_run.italic = True

In [5]:
def add_internal_reference_table(document):
    centered_heading = document.add_paragraph("Table only for internal reference")
    centered_heading.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    centered_heading.runs[0].font.size = Pt(12)
    centered_heading.runs[0].font.name = 'Times New Roman'
    headings = ["Patient name", "Age", "Gender", "Diagnosis", "Comorbidities", "Specimen",
                "Histopathology in case of breast cancer/ovarian cancer etc.", "Metastatic status",
                "Known Infectious causes (for example HPV)?", "Treatment and progression/resistance history",
                "TMB status", "MSI status", "PD-L1 status", "HRR status", "MMR status",
                "Genes for clinical interpretation", "Additional comment/Family history etc."]
    table = document.add_table(rows=0, cols=2)
    table.style = 'Table Grid'
    #table.autofit = False
    #table.allow_autofit = False
    table.columns[0].width = Cm(5.26)
    table.columns[1].width = Cm(12.7)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    for heading in headings:
        row = table.add_row().cells
        row[0].text = heading
        row[1].text = ""
        for cell in row:
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.font.name = 'Times New Roman'
                    run.font.size = Pt(12)
                    
    # Add one line spacing after the table
    document.add_paragraph('').add_run().add_break(WD_BREAK.LINE)
    
    # Add biomarkers_df table
    column_widths = [28 / biomarkers_df.shape[1]] * biomarkers_df.shape[1]
    biomarkers_table = document.add_table(biomarkers_df.shape[0] + 1, biomarkers_df.shape[1])
    biomarkers_table.style = 'Table Grid'
    biomarkers_table.autofit = False
    biomarkers_table.allow_autofit = False
    biomarkers_table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    
    # Set the width for each column
    for j, width in enumerate(column_widths):
        biomarkers_table.columns[j].width = Cm(width)
    
    # Add column headings
    for j, col in enumerate(biomarkers_df.columns):
        cell = biomarkers_table.cell(0, j)
        cell.text = col
        cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell.paragraphs[0].runs[0].font.size = Pt(10)
        cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    
    # Add data to the table
    for i in range(biomarkers_df.shape[0]):
        for j in range(biomarkers_df.shape[1]):
            cell = biomarkers_table.cell(i + 1, j)
            cell.text = str(biomarkers_df.iloc[i, j])
            cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
            cell.paragraphs[0].runs[0].font.size = Pt(9)

    # Add page break after the table
    document.add_page_break()

In [6]:
def patient_information_table():
    sections = doc.sections
    for section in sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(2)
        section.right_margin = Cm(2)
    FFPE_block_number = patient_info.get('FFPE_block_number', '') 
    FFPE = 'FFPE Block ' + '( ' + FFPE_block_number + ')' if FFPE_block_number else ''
    table = doc.add_table(rows=1, cols=2)
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(5.1)
    table.columns[1].width = Cm(11.89)
    table.style = 'Table Grid'
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.cell(0, 0).text = 'Name:'
    table.cell(0, 1).text = patient_info.get('patient_name', '')or '' 
    table.add_row().cells[0].text = 'Age/Gender:'
    table.cell(1, 1).text = f"{patient_info.get('patient_age', '')} years /{patient_info.get('patient_gender', '')}"or '' 
    table.add_row().cells[0].text = 'Referring Physician:'
    table.cell(2, 1).text = patient_info.get('referring_physician', '')or '' 
    table.add_row().cells[0].text = 'Referring Centre:'
    table.cell(3, 1).text = patient_info.get('referring_centre', '')or '' 
    table.add_row().cells[0].text = 'Specimen Type:'
    table.cell(4, 1).text = FFPE
    table.add_row().cells[0].text = 'Sample ID:'
    table.cell(5, 1).text = patient_info.get('sample_id', '')or '' 
    table.add_row().cells[0].text = 'Patient ID:'
    table.cell(6, 1).text = patient_info.get('patient_id', '')or '' 
    table.add_row().cells[0].text = 'Date received:'
    table.cell(7, 1).text = patient_info.get('date_received', '')or '' 
    table.add_row().cells[0].text = 'Report Date:'
    table.cell(8, 1).text = patient_info.get('report_date', '')or ''
    
    font_style_column1 = 'Times New Roman'
    font_size_column1 = Pt(12)
    bold_column1 = True
    font_style_column2 = 'Times New Roman'
    font_size_column2 = Pt(12)
    bold_column2 = False
    for row in table.rows:
        row.cells[0].paragraphs[0].runs[0].font.name = font_style_column1
        row.cells[0].paragraphs[0].runs[0].font.size = font_size_column1
        row.cells[0].paragraphs[0].runs[0].bold = bold_column1
    for row in table.rows:
        row.cells[1].paragraphs[0].runs[0].font.name = font_style_column2
        row.cells[1].paragraphs[0].runs[0].font.size = font_size_column2
        row.cells[1].paragraphs[0].runs[0].bold = bold_column2

In [7]:
def patient_information(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(7.69)
    new_height = Cm(0.8)
    paragraph = document.add_paragraph()
    paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", "Patient_information.png")
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = doc.add_paragraph()
    p.paragraph_format.line_spacing = 1
    p.paragraph_format.space_after = 12

    patient_information_table()


In [8]:
def oncoprecise_image(document):
    p = document.add_paragraph()
    paragraph = document.add_paragraph()
    paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    new_width = Cm(14.31)
    new_height = Cm(1.27)
    image_path = os.path.join("Heading_Images", 'Oncoprecise.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)

In [9]:
def clinical_history(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(6.73)
    new_height = Cm(0.63)
    paragraph = document.add_paragraph()
    paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Clinical_history.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)

    # Data from Unipath file
    clinical_details_text = data.get("Clinical Details", "")
    clinical_details_text = clinical_details_text.replace("\n", "")
    p = document.add_paragraph()
    p.add_run(clinical_details_text)
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'

In [10]:
# Data from Unipath file
result_summary_df['Alteration Description '] = result_summary_df['Alteration Description '].str.lower()

# Extract TMB information
tmb_status = ""
tmb_score = ""
loh = ""
msi_status = ""
msi_score = ""
cnvs_findings = ""

for index, row in result_summary_df.iterrows():
    if "tumor mutational burden" in row["Alteration Description "]:
        tmb_status = row['Findings '].strip()
    if "tmb score" in row["Alteration Description "]:
        tmb_score = row['Findings '].strip()
    if "microsatellite instability" in row["Alteration Description "]:
        msi_status = row["Findings "].strip()
    if "msi score" in row["Alteration Description "]:
        msi_score = row["Findings "].strip()
    if 'loss of heterozygosity' in row["Alteration Description "]:
        loh = row["Findings "].strip()
    if 'copy number variants' in row["Alteration Description "]:
        cnvs = row["Findings "].strip()
        cnvs =str(cnvs).lower()
        if 'not identified' in cnvs:
            cnvs_findings = row["Findings "].strip()
        else:
            cnvs_findings = 'cnv_df should be checked'
            cnv_df.columns = cnv_df.columns.str.strip()
            cnv_df['gene_mutation_list'] = cnv_df.apply(lambda row: f"{row['Gene'].strip()} ({row['Locus'].split()[1]})x{row['Copy Number'].strip()}", axis=1)
            cnvs_findings = '\n'.join(cnv_df['gene_mutation_list'])
            
    if "snvs and indels" in row["Alteration Description "]:
        snvs = row["Findings "].strip()
        snvs =str(snvs).lower()
        if 'not identified' in snvs:
            snvs_findings = row["Findings "].strip()
        else: 
            snv_df.columns = snv_df.columns.str.strip()
            snv_df['gene_mutation_list'] = snv_df.apply(lambda row: f"{row['Gene/Transcript'].split()[0]} ({row['Variant/Amino Acid Change'].split()[0]})", axis=1)
            snvs_findings = '\n'.join(snv_df['gene_mutation_list'])
            
    if "novel and known fusions identified" in row["Alteration Description "]:
        fusion = row["Findings "].strip()
        fusion =str(fusion).lower()
        if 'not identified' in fusion:
            fusion_findings = row["Findings "].strip()
        else:
            fusion_findings = 'fusion_df should be checked'
            fusion_df.columns = fusion_df.columns.str.strip()
        
        
combined_tmb_value = f"TMB-{tmb_status} ({tmb_score} mut/mb)" 
print(combined_tmb_value)
combined_msi_value = f"{msi_status} (Score:{msi_score})" 
print(combined_msi_value)
loh_score = f"LOH({loh})"

# Consolidate the results into a dictionary
table_data = {
    "Gene Mutations": snvs_findings,
    "Gene Fusions": fusion_findings,
    "Copy number variants (CNVs)": cnvs_findings,
    "Loss of Heterozygosity (LOH)": loh_score,
    "Tumor mutational burden (TMB)": combined_tmb_value,
    "Microsatellite instability (MSI)": combined_msi_value,}

def report_summary_table(document, data):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0.14)
        section.right_margin = Cm(2.5)
    table = document.add_table(rows=1, cols=2)
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(7.51)
    table.columns[1].width = Cm(9.97)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.style = 'Table Grid'
    first_row = table.rows[0].cells
    first_row[0].text = "Alteration Description"
    first_row[1].text = "Findings"
    for cell in first_row:
        cell.paragraphs[0].runs[0].bold = True
        cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell.paragraphs[0].runs[0].font.size = Pt(12)
        cell.paragraphs[0].runs[0].font.color.rgb = RGBColor(0, 0, 128) 
    row_height_cm = 1.04
    first_row = table.rows[0]
    first_row.height = Pt(row_height_cm * 28.3465)
    for key, value in data.items():
        row_cells = table.add_row().cells
        cell = row_cells[0]
        cell.text = key
        cell.paragraphs[0].runs[0].bold = True
        cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell.paragraphs[0].runs[0].font.size = Pt(12)
        row_cells[1].text = value
        cell1 = row_cells[1]
        cell1.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell1.paragraphs[0].runs[0].font.size = Pt(12)
    return table

TMB-LOW (4.74 mut/mb)
MSI STABLE-(MSS) (Score:10.34)


In [11]:
def report_summary(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(5.82)
    new_height = Cm(0.66)
    paragraph = document.add_paragraph()
    image_path = os.path.join("Heading_Images", 'Report_summary.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2 * 0.42)
    
    FFPE=patient_info.get('FFPE_block_number', '')or '___' 
    tumor_cell=patient_info.get('tumor_cells', '')or '___'
    tumor_cell = tumor_cell+'%'
    p.add_run("The test was performed on Block ")
    p.add_run(FFPE).bold = True
    p.add_run(" which showed presence of ")
    p.add_run(tumor_cell).bold = True
    p.add_run(" tumor cells.")    
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'

    report_summary_table(document, table_data)
    document.paragraphs[-1].paragraph_format.space_before = Pt(0)
    note1 = document.add_paragraph("*Note: This High TMB Score might be affected by higher deamination in this sample which may be due to the intrinsic nature of FFPE tissue.")
    note1.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note1.paragraph_format.left_indent = Cm(2.5)
    note1.paragraph_format.right_indent = Cm(2.5)
    note1.paragraph_format.space_after = Cm(0)
    note1.paragraph_format.line_spacing = 1
    for run in note1.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'
    note2 = document.add_paragraph("**Note: Assessment of HRR/HRD using orthogonal methods is recommended.")
    note2.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note2.paragraph_format.left_indent = Cm(2.5)
    note2.paragraph_format.right_indent = Cm(2.5)
    note1.paragraph_format.space_after = Cm(0)
    for run in note2.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'

In [12]:
def therapeutic_summary_table(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(2)
        section.right_margin = Cm(2)
        
    line1_paragraph = document.add_paragraph("Therapeutic summary is relevant for the patient’s diagnosis")
    line1_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    line1_paragraph.paragraph_format.left_indent = Cm(2.5)
    line1_paragraph.paragraph_format.right_indent = Cm(2.5)
    line1_paragraph.paragraph_format.space_after = Cm(0)
    line2_paragraph = document.add_paragraph("(Additional in-trial therapies relevant for this patient are listed in the 'clinical trials' section)")
    line2_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    line2_paragraph.paragraph_format.left_indent = Cm(2.5)
    line2_paragraph.paragraph_format.right_indent = Cm(2.5)
    line2_paragraph.paragraph_format.space_after = Cm(0)
    for run in line1_paragraph.runs + line2_paragraph.runs:
        run.font.size = Pt(12)
        run.font.name = 'Times New Roman'
    paragraph = document.add_paragraph() 
    table = document.add_table(rows=10, cols=6)  
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(1.5)
    table.columns[1].width = Cm(2.5)
    table.columns[2].width = Cm(1.5)
    table.columns[3].width = Cm(5.96)
    table.columns[4].width = Cm(4)
    table.columns[5].width = Cm(3.75)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.style = 'Table Grid'

    header = table.rows[0].cells
    header[0].text = "Tier"
    header[1].text = "Variant/ Biomarker"
    header[2].text = "VAF"
    header[3].paragraphs[0].add_run("Therapy (Approved/ In-trial#) \n (Please see details in the subsequent sections of the report)")
    header[4].paragraphs[0].add_run("Potential drug resistance \n (Please see details in “Drug resistance” sections)")
    header[5].paragraphs[0].add_run("Prognostic \n (Please see details in “Prognostic significance” sections)")

    for cell in header:
        cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        cell.paragraphs[0].runs[0].font.size = Pt(12)
        cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell.paragraphs[0].runs[0].bold = True

    for row in table.rows[1:]:
        for cell in row.cells:
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.font.size = Pt(10)
                    run.font.name = 'Times New Roman'   
    return table 

In [13]:
def therapeutic_summary(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(12.95)
    new_height = Cm(0.57)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Therapeutic_summary.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    therapeutic_summary_table(document)
    document.paragraphs[-1].paragraph_format.space_before = Pt(0)
    note1 = document.add_paragraph("Note 1: The therapies in bold have been approved by the FDA. Further information about these therapies can be found in the “Immunotherapy” and “Gene and Variant description” section of this report.")
    note1.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note1.paragraph_format.left_indent = Cm(2.5)
    note1.paragraph_format.right_indent = Cm(2.5)
    note1.paragraph_format.space_after = Cm(0)
    note1.paragraph_format.line_spacing = 1
    for run in note1.runs:
        run.font.size = Pt(10)
        run.font.name = 'Times New Roman'
    note2 = document.add_paragraph("Note 2: The therapies labeled as # are being tested in registered clinical trials for which the patient may be eligible to enroll. Please see the “Clinical trials” section for more details.")
    note2.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note2.paragraph_format.left_indent = Cm(2.5)
    note2.paragraph_format.right_indent = Cm(2.5)
    note1.paragraph_format.space_after = Cm(0)
    for run in note2.runs:
        run.font.size = Pt(10)
        run.font.name = 'Times New Roman'
    note3 = document.add_paragraph("Note 3: The *Drug resistance associations mentioned in the above table may be observed in preclinical or clinical studies. More details can be found in the “Drug resistance” section for each variation.")
    note3.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note3.paragraph_format.left_indent = Cm(2.5)
    note3.paragraph_format.right_indent = Cm(2.5)
    note2.paragraph_format.space_after = Cm(0)
    for run in note3.runs:
        run.font.size = Pt(10)
        run.font.name = 'Times New Roman'
    note4 = document.add_paragraph("Note 4: Additional therapeutic information and the prognostic information can be found in the “variant details” and the “clinical trials” section of the report.")
    note4.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note4.paragraph_format.left_indent = Cm(2.5)
    note4.paragraph_format.right_indent = Cm(2.5)
    note3.paragraph_format.space_after = Cm(0)
    for run in note4.runs:
        run.font.size = Pt(10)
        run.font.name = 'Times New Roman'

In [14]:
pdl1_status=patient_info.get('PD-L1_status', '')or ''
pdl1_status_score=patient_info.get('PD-L1_status_score', '')or ''
combined_pdl1_value = f"{pdl1_status}({pdl1_status_score})" 
mrr_genes = ['MSH2', 'MSH6', 'MLH1', 'PMS2'] 
biomarkers_df['Gene'] = biomarkers_df['Genomic Alteration '].str.split().str[0]
def check_MRR_genes(df, genes):
    found_genes = []
    for gene in genes:
        if gene in df['Gene'].values:
            found_genes.append(gene)
    return found_genes
        
if 'Gene' in biomarkers_df.columns:
    found_genes = check_MRR_genes(biomarkers_df, mrr_genes)
else:
    found_genes = []

if found_genes:
    mmr_status = "Loss-of-function mutation was found in the MMR gene/s: " + ", ".join(found_genes) + '\n' + "IHC is recommended to confirm the nuclear loss of MMR proteins"
else:
    mmr_status = f"No loss-of-function mutation in the 4 MMR genes: {', '.join(mrr_genes)}"
    
def add_formatted_paragraph(doc, text):
    paragraph = doc.add_paragraph()
    run = paragraph.add_run(text)
    run.font.name = 'Times New Roman'
    run.font.size = Pt(12)
    paragraph.paragraph_format.line_spacing = 1.0
    paragraph.paragraph_format.left_indent = Cm(2.5)
    paragraph.paragraph_format.right_indent = Cm(2.5)
    return paragraph
def immunotherapy_markers(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(10.27)
    new_height = Cm(0.66)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Immunotherapy_markers.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)

    add_formatted_paragraph(document, f"1- PDL1 by IHC (SP263)- {combined_pdl1_value}")
    add_formatted_paragraph(document, f"2- Tumor Mutation Burden- {combined_tmb_value}")
    add_formatted_paragraph(document, f"3- Microsatellite Instability- {combined_msi_value}")
    add_formatted_paragraph(document, f"4- MMR (Mismatch repair) status- {mmr_status}")

In [15]:
if mmr_status == 'No loss-of-function mutation in the 4 MMR genes: MSH2, MSH6, MLH1, PMS2':
    mmr_status_result = 'likely MMR proficient'
else:
    mmr_status_result = 'likely MMR deficient'
    
if 'MSI STABLE' in msi_status:
    msi_status_result = 'MSS'
else:
    msi_status_result = 'MSI-High'

def add_hyperlink(paragraph, text, url):
    part = paragraph.part
    r_id = part.relate_to(url, docx.opc.constants.RELATIONSHIP_TYPE.HYPERLINK, is_external=True)
    hyperlink = docx.oxml.shared.OxmlElement('w:hyperlink')
    hyperlink.set(docx.oxml.shared.qn('r:id'), r_id)
    new_run = docx.text.run.Run(docx.oxml.shared.OxmlElement('w:r'), paragraph)
    new_run.text = text
    new_run.style = get_or_create_hyperlink_style(part.document)
    hyperlink.append(new_run._element)
    paragraph._p.append(hyperlink)
    return hyperlink

def get_or_create_hyperlink_style(document):
    if "Hyperlink" not in document.styles:
        if "Default Character Font" not in document.styles:
            default_character_font = document.styles.add_style("Default Character Font", docx.enum.style.WD_STYLE_TYPE.CHARACTER, True)
            default_character_font.element.set(docx.oxml.shared.qn('w:default'), "1")
            default_character_font.priority = 1
            default_character_font.hidden = True
            default_character_font.unhide_when_used = True
            del default_character_font
        hyperlink_style = document.styles.add_style("Hyperlink", docx.enum.style.WD_STYLE_TYPE.CHARACTER, True)
        hyperlink_style.base_style = document.styles["Default Character Font"]
        hyperlink_style.unhide_when_used = True
        hyperlink_style.font.color.rgb = docx.shared.RGBColor(0x05, 0x63, 0xC1)
        hyperlink_style.font.underline = True
        del hyperlink_style
    return "Hyperlink"

def approved_immunotherapy(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
        
    new_width = Cm(9.58)
    new_height = Cm(0.69)
    
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Approved_immunotherapy.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    main_tumor_types = patient_info.get('Main Tumor Type', '')
    p = document.add_paragraph()        
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run(f"This is a PD-L1 {pdl1_status}, ")
    p.add_run(f"TMB-{tmb_status}, ")
    p.add_run(f"{msi_status_result}, ")
    p.add_run(f"{mmr_status_result} case of {main_tumor_types}")  
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman' 

    tumor_types = patient_info.get('Main Tumor Type', '').split(', ')
    fda_approved_df = pd.read_excel('V2 running FDA-approved cancer Immunotherapies Jan 2024.xlsx')
    filtered_df = fda_approved_df[fda_approved_df['Main tumor type'].str.contains('Solid tumours', case=False)]
    for tumor_type in tumor_types:
        matching_rows2 = fda_approved_df[fda_approved_df['Main tumor type'].str.contains(fr'{tumor_type}', case=False, regex=True, na=False, flags=re.IGNORECASE)]        
        for index, row in matching_rows2.iterrows():
            drug_name = row['Immunotherapy']
            link_holder = document.add_paragraph()
            link_holder.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            link_holder.paragraph_format.left_indent = Cm(2.5)
            link_holder.paragraph_format.right_indent = Cm(2.5)
            drug=drug_name.replace(" ","+")
            add_hyperlink(link_holder, drug_name, f"https://www.google.com/search?q={drug}+mechanism+of+action")
            main_tumor_type=row['Main tumor type']
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run(main_tumor_type)
            for run in p.runs:
                run.font.size = Pt(12)
                run.font.name ='Times New Roman'
            detailed_tumor_type=row['FDA-approved indications']
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run(detailed_tumor_type)
            for run in p.runs:
                run.font.size = Pt(12)
                run.font.name ='Times New Roman'
            biomarker_details = row['Biomarker']
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run(str(biomarker_details))
            for run in p.runs:
                run.font.size = Pt(12)
                run.font.name ='Times New Roman'
            details_of_approval = row['Details of approval']
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run(details_of_approval)
            for run in p.runs:
                run.font.size = Pt(12)
                run.font.name ='Times New Roman'

    matching_rows3 = pd.DataFrame()
    if mmr_status_result == 'likely MMR deficient':
        matching_rows3 = pd.concat([matching_rows3, filtered_df[filtered_df['Biomarker'].str.contains('dMMR')]])
    if msi_status_result == 'MSI-High':
        matching_rows3 = pd.concat([matching_rows3, filtered_df[filtered_df['Biomarker'].str.contains('MSI-High')]])
    # Extract 'TMB-High' from 'Biomarkers' if mutational_burden_status == 'High'
    if tmb_status == 'High':
        matching_rows3 = pd.concat([matching_rows3, filtered_df[filtered_df['Biomarker'].str.contains('TMB-High')]])
    matching_rows3.drop_duplicates(inplace=True)
    for index, row in matching_rows3.iterrows():
        drug_name = row['Immunotherapy']
        link_holder = document.add_paragraph()
        link_holder.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        link_holder.paragraph_format.left_indent = Cm(2.5)
        link_holder.paragraph_format.right_indent = Cm(2.5)
        drug=drug_name.replace(" ","+")
        add_hyperlink(link_holder, drug_name, f"https://www.google.com/search?q={drug}+mechanism+of+action")
        main_tumor_type=row['Main tumor type']
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run(main_tumor_type)
        for run in p.runs:
            run.font.size = Pt(12)
            run.font.name ='Times New Roman'
        detailed_tumor_type=row['FDA-approved indications']
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run(detailed_tumor_type)
        for run in p.runs:
            run.font.size = Pt(12)
            run.font.name ='Times New Roman'
        biomarker_details = row['Biomarker']
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run(biomarker_details)
        for run in p.runs:
            run.font.size = Pt(12)
            run.font.name ='Times New Roman'
        details_of_approval = row['Details of approval']
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run(details_of_approval)
        for run in p.runs:
            run.font.size = Pt(12)
            run.font.name ='Times New Roman'

In [16]:
def intrial_immunotherapy(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(9.58)
    new_height = Cm(0.56)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Intrial_immunotherapy.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    paragraph = document.add_paragraph("Multiple immunotherapies are being investigated in clinical trials for this patient’s genomic profile and/or diagnosis (Please see “Clinical trials” section for more details).")
    paragraph.paragraph_format.left_indent = Cm(2.5)
    paragraph.paragraph_format.right_indent = Cm(2.5)
    run_text = paragraph.runs[0]
    font = run_text.font
    font.name = 'Times New Roman'
    font.size = Pt(12)

In [17]:
# Data from Unipath file
try:
    snv_df.rename(columns={'Gene/Transcript': 'Gene'}, inplace=True)
    snv_df.rename(columns={'Variant/Amino Acid Change': 'Genomic alteration'}, inplace=True)
    snv_df.rename(columns={'Variant classification': 'Clinical Significance'}, inplace=True)
    snv_df.rename(columns={'TIER classification': 'AMP^ Classification'}, inplace=True)
    snv_df.rename(columns={'Total Coverage/VAF': 'Variant Allelic Frequency (VAF%)'}, inplace=True)
    snv_df['Variant Allelic Frequency (VAF%)'] = snv_df['Variant Allelic Frequency (VAF%)'].str.replace(r'\d+X\s*', '', regex=True)
    snv_df1 = snv_df.drop(['Locus', 'gene_mutation_list'], axis=1)
except AttributeError:
    snv_df1 = None
  
try:
    fusion_df.rename(columns={'Gene/Transcript ': 'Gene'}, inplace=True)
    fusion_df1 = fusion_df
except AttributeError: 
    fusion_df1 = None

try:
    cnv_df1 = cnv_df.drop(['gene_mutation_list'], axis=1)
except AttributeError:
    cnv_df1 = None

def variant_detail_table1(document, snv_df1, fusions_df1, cnv_df1):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(1.5)
        section.right_margin = Cm(2)
    
    tables = [] 
    # SNV Table
    if snv_df1 is not None and not snv_df1.empty:
        table = document.add_table(rows=snv_df1.shape[0] + 1, cols=snv_df1.shape[1])
        table.style = 'Table Grid'
        table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        for col_num, col_name in enumerate(snv_df1.columns):
            cell = table.cell(0, col_num)
            cell.text = col_name
            run = cell.paragraphs[0].runs[0]
            run.bold = True
            run.font.name = 'Times New Roman'
            run.font.size = Pt(10)
            cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        for row_num in range(snv_df1.shape[0]):
            for col_num, value in enumerate(snv_df1.iloc[row_num]):
                cell = table.cell(row_num + 1, col_num)
                cell.text = str(value)
                run = cell.paragraphs[0].runs[0]
                run.font.name = 'Times New Roman'
                run.font.size = Pt(10)
                cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        if snv_df1.shape[0] < len(table.rows):
            last_row = table.rows[-1]
            if all(cell.text == '' for cell in last_row.cells):
                table._element.remove(last_row._element)
        tables.append(table)  # Append table to the list
    
    # Fusion Table
    if fusions_df1 is not None and not fusions_df1.empty:
        try:
            table = document.add_table(rows=fusions_df1.shape[0] + 1, cols=fusions_df1.shape[1])
            table.style = 'Table Grid'
            table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            for col_num, col_name in enumerate(fusions_df1.columns):
                cell = table.cell(0, col_num)
                cell.text = col_name
                run = cell.paragraphs[0].runs[0]
                run.bold = True
                run.font.name = 'Times New Roman'
                run.font.size = Pt(10)
                cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            for row_num in range(fusions_df1.shape[0]):
                for col_num, value in enumerate(fusions_df1.iloc[row_num]):
                    cell = table.cell(row_num + 1, col_num)
                    cell.text = str(value)
                    run = cell.paragraphs[0].runs[0]
                    run.font.name = 'Times New Roman'
                    run.font.size = Pt(10)
                    cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            if fusions_df1.shape[0] < len(table.rows):
                last_row = table.rows[-1]
                if all(cell.text == '' for cell in last_row.cells):
                    table._element.remove(last_row._element)
            tables.append(table)  
        except NameError:
            fusions_df1 = None

    # CNV Table
    if cnv_df1 is not None and not cnv_df1.empty:
        table = document.add_table(rows=cnv_df1.shape[0] + 1, cols=cnv_df1.shape[1])
        table.style = 'Table Grid'
        table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        for col_num, col_name in enumerate(cnv_df1.columns):
            cell = table.cell(0, col_num)
            cell.text = col_name
            run = cell.paragraphs[0].runs[0]
            run.bold = True
            run.font.name = 'Times New Roman'
            run.font.size = Pt(10)
            cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        for row_num in range(cnv_df1.shape[0]):
            for col_num, value in enumerate(cnv_df1.iloc[row_num]):
                cell = table.cell(row_num + 1, col_num)
                cell.text = str(value)
                run = cell.paragraphs[0].runs[0]
                run.font.name = 'Times New Roman'
                run.font.size = Pt(10)
                cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        if cnv_df1.shape[0] < len(table.rows):
            last_row = table.rows[-1]
            if all(cell.text == '' for cell in last_row.cells):
                table._element.remove(last_row._element)
        tables.append(table)  
    
    return tables 
  


In [18]:
def variant_detail_table2(document, variants):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(1.5)
        section.right_margin = Cm(2)
    
    table = document.add_table(rows=4, cols=9)  
    table.autofit = False
    table.style = 'Table Grid'
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    
    widths = [2, 1.5, 1.5, 2, 1.5, 1.5, 1.5, 1.5, 1.5]  
    for i, width in enumerate(widths):
        table.cell(0, i).width = Cm(width)

    heights = [1, 1, 1, 1]
    for i, height in enumerate(heights):
        table.rows[i].height = Cm(height)
        
    header = table.rows[0].cells
    header[0].text = "Variant"
    table.cell(0, 0).merge(table.cell(1, 0))
    header[1].text = "dbSNP\nrs ID"
    table.cell(0, 1).merge(table.cell(1, 1))
    header[2].text = "COSMIC"
    table.cell(0, 2).merge(table.cell(1, 2))
    header[3].text = "ClinVar"
    table.cell(0, 3).merge(table.cell(1, 3))
    header[4].text = "In-silico predictors"
    table.cell(0, 4).merge(table.cell(0, 6))
    header[7].text = "Population Database"
    table.cell(0, 7).merge(table.cell(0, 8))
    
    second_row = table.rows[1].cells
    second_row[0].text = "Variant"
    second_row[1].text = "dbSNP\nrs ID"
    second_row[2].text = "COSMIC"
    second_row[3].text = "ClinVar"
    second_row[4].text = "SIFT"
    second_row[5].text = "Polyphen"
    second_row[6].text = "MT2"
    second_row[7].text = "gnomAD"
    second_row[8].text = "ExAc"
    
    header = table.rows[0].cells
    second_row = table.rows[1].cells
    for i in range(min(len(header), len(second_row), 9)):
        header[i].paragraphs[0].runs[0].font.bold = True
        header[i].paragraphs[0].runs[0].font.name = 'Times New Roman'
        header[i].paragraphs[0].runs[0].font.size = Pt(11)
        header[i].paragraphs[0].paragraph_format.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        header[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER

        second_row[i].paragraphs[0].runs[0].font.bold = True
        second_row[i].paragraphs[0].runs[0].font.name = 'Times New Roman'
        second_row[i].paragraphs[0].runs[0].font.size = Pt(11)
        second_row[i].paragraphs[0].paragraph_format.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        second_row[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
        
    total_rows_needed = max(len(snv_df1), len(variants)) + 2  
    while len(table.rows) < total_rows_needed:
        table.add_row()
    for i, (index, data) in enumerate(snv_df1.iterrows()):
        row = table.rows[i + 2]
        row.cells[0].text = data.get("Gene", "")
        
    for i, variant in enumerate(variants):
        row = table.rows[i + 2]
        row.cells[1].text = variant.get("rs_id", "")
        row.cells[2].text = variant.get("cosmic_id", "")
        row.cells[3].text = variant.get("clinvar_id", "")
        row.cells[4].text = variant["predictors"].get("SIFT", "")
        row.cells[5].text = variant["predictors"].get("Polyphen", "")
        row.cells[6].text = variant["predictors"].get("MT2", "")
        row.cells[7].text = variant['population_database'].get('gnomAD', "")
        row.cells[8].text = variant['population_database'].get('ExAc', "")

    for row in table.rows[2:]:
        for cell in row.cells:
            cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            for paragraph in cell.paragraphs:
                if paragraph.runs and paragraph.runs[0].text.strip():
                    paragraph.runs[0].font.size = Pt(10)
                    paragraph.runs[0].font.name = 'Times New Roman'
                    paragraph.paragraph_format.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT

    return table

In [19]:
def variant_details(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(9.58)
    new_height = Cm(0.56)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Variant_details.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("Table-1: List of variants identified in this sample").bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    variant_detail_table1(document, snv_df1, fusion_df1, cnv_df1)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("Refer to supplementary information for the classification criteria details. ^")
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("Table-2: Significance of Variants reported in database").bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    variants = [ { "rs_id": "",
        "cosmic_id": "",
        "clinvar_id": "",
        "predictors": {"SIFT": "", "Polyphen": "",  "MT2": "",},
        "population_database": { "gnomAD": "", "ExAc": "",}},
      {"rs_id": "",
        "cosmic_id": "",
        "clinvar_id":"",
        "predictors": {"SIFT": "", "Polyphen": "", "MT2": "", },
        "population_database": {"gnomAD": "", "ExAc": "", }}]
    variant_detail_table2(document, variants)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("P- Pathogenic, D- Damaging, B-Benign")
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman'
    p = document.add_paragraph()

In [20]:
def extract_variant_info(snv_df, cnv_df, variants_description):
    gene_variants_data = {'Gene_name': pd.Series([], dtype='object'), 'Variant': pd.Series([], dtype='object')}
    genes_involved = []
    
    if snv_df is not None:
        snv_genes = snv_df['Gene_Name'].str.upper().unique()
        genes_involved.extend(snv_genes)
    
    if cnv_df is not None:
        cnv_genes = cnv_df['Gene_Name'].str.upper().unique()
        genes_involved.extend(cnv_genes)

    genes_involved = list(set(genes_involved))

    for variant in variants_description:
        gene_name = variant.split(":")[0].upper()
        if gene_name in genes_involved:
            gene_variants_data['Gene_name'] = pd.concat([gene_variants_data['Gene_name'], pd.Series([gene_name], dtype='object')], ignore_index=True)
            gene_variants_data['Variant'] = pd.concat([gene_variants_data['Variant'], pd.Series([variant], dtype='object')], ignore_index=True)

    gene_variants_data_df = pd.DataFrame(gene_variants_data)
    print('------------------------------------------------------')
    print(gene_variants_data_df)
    gene_variants_data_df.to_excel('alter_variant_data.xlsx', index=False)

In [21]:
def add_hyperlink(paragraph, drug_name, url):
    part = paragraph.part
    r_id = part.relate_to(url, docx.opc.constants.RELATIONSHIP_TYPE.HYPERLINK, is_external=True)
    hyperlink = docx.oxml.shared.OxmlElement('w:hyperlink')
    hyperlink.set(docx.oxml.shared.qn('r:id'), r_id)
    new_run = docx.text.run.Run(docx.oxml.shared.OxmlElement('w:r'), paragraph)
    new_run.text = drug_name
    new_run.style = get_or_create_hyperlink_style(part.document)
    hyperlink.append(new_run._element)
    paragraph._p.append(hyperlink)
    return hyperlink
    
def get_or_create_hyperlink_style(document):
    if "Hyperlink" not in document.styles:
        if "Default Character Font" not in document.styles:
            default_character_font = document.styles.add_style("Default Character Font", docx.enum.style.WD_STYLE_TYPE.CHARACTER, True)
            default_character_font.element.set(docx.oxml.shared.qn('w:default'), "1")
            default_character_font.priority = 1
            default_character_font.hidden = True
            default_character_font.unhide_when_used = True
            del default_character_font
        hyperlink_style = document.styles.add_style("Hyperlink", docx.enum.style.WD_STYLE_TYPE.CHARACTER, True)
        hyperlink_style.base_style = document.styles["Default Character Font"]
        hyperlink_style.unhide_when_used = True
        hyperlink_style.font.color.rgb = docx.shared.RGBColor(0x05, 0x63, 0xC1)
        hyperlink_style.font.underline = True
        del hyperlink_style
    return "Hyperlink"   

def extract_hyperlink_data(drug_des):
    pattern = re.compile(r'Drug: (\w+)', re.IGNORECASE)
    matches = re.findall(pattern, drug_des)
    return matches
    
def set_run_color(run, color):
    font = run.font
    font.color.rgb = color
    
def generate_fusion_combinations(gene_names):
    fusion_combinations = set()
    fusion_combinations.add(gene_names[0] + ' Fusion')
    fusion_combinations.add(gene_names[1] + ' Fusion')
    fusion_combinations.add('-'.join(gene_names) + ' Fusion')
    fusion_combinations.add('-'.join(reversed(gene_names)) + ' Fusion')
    return fusion_combinations

def add_gene_summary_text(document, gene_heading, ncbi_summary, gene_roles, gene_ckb_des, my_cancer_des, altered_var, oncokb, prognosis_pubmed_des, prognosis_germline_des, approved_therapy_des, drug_des):
    #Gene Heading
    p = document.add_paragraph()
    p.add_run(gene_heading).bold = True
    p.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    for run in p.runs:
        run.font.name = 'Times New Roman'
        run.font.size = Pt(12) 
    #Gene Summary
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("Gene Summary").bold = True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name = 'Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    if pd.isna(ncbi_summary):
        ncbi_summary = 'NCBI summary information could not be found in the in-house database'
    ncbi_summary = re.sub(r'\[provided by RefSeq, [A-Za-z]+ \d+\]', '', ncbi_summary)
    p.add_run(ncbi_summary)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'
        
    #Altered variant
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("Altered variant").bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run(altered_var)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'

    p = document.add_paragraph('Link(s) to OncoKB for the variant:\n')
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    oncokb = str(oncokb)
    hyperlink_text = oncokb.replace('-', '/').replace(' ', '/')
    hyperlink_url = f'https://www.oncokb.org/gene/{hyperlink_text}'
    add_hyperlink(p, oncokb, hyperlink_url)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'

    if 'Fusion' in gene_heading:
        gene_names = gene_heading.replace(' Fusion', '').split('-')
        fusion_combinations = generate_fusion_combinations(gene_names)
        fusion_combinations.discard(' '.join(gene_names) + ' Fusion')
        fusion_combinations.discard(' '.join(reversed(gene_names)) + ' Fusion')
        # Sort fusion combinations by length and then alphabetically
        fusion_combinations = sorted(fusion_combinations, key=lambda x: (len(x.split()[0]), x))
        
        for fusion_heading in fusion_combinations:
            hyperlink_text = fusion_heading.replace('-', '/').replace(' ', '/')
            hyperlink_url = f'https://www.oncokb.org/gene/{hyperlink_text}'
            add_hyperlink(p, fusion_heading, hyperlink_url)
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p = doc.add_paragraph('\n')
        for run in p.runs:
            run.font.size = Pt(11)
            run.font.name = 'Times New Roman'
    
    # Gene disease association
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Gene -Disease association').bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run(gene_roles)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    if pd.isna(gene_ckb_des):
        gene_ckb_des = 'CKB-core information could not be found in the in-house database'
    p.add_run(gene_ckb_des)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    if pd.isna(my_cancer_des):
        my_cancer_des = 'There is no My cancer genome information for this gene'
    else:
        my_cancer_des = re.sub(r'\[(\d+)\]', r'[My cancer genome]', my_cancer_des)
        my_cancer_des = str(my_cancer_des)
    p.add_run(my_cancer_des)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman'

    #Prognostic significance
    if prognosis_pubmed_des is not None:
        prognosis_pubmed_des = str(prognosis_pubmed_des)
        pattern = re.compile(r'\(PMID: \d+\)\.')
        prognosis_pubmed_des = re.sub(pattern, lambda x: x.group() + '\n', prognosis_pubmed_des)
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run('Prognostic significance').bold=True
        for run in p.runs:
            run.font.size = Pt(12)
            run.font.name ='Times New Roman'
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        if prognosis_pubmed_des.lower() == 'nan':
            prognosis_pubmed_des = 'Prognostic significance information could not be found in the in-house database'
        p.add_run(prognosis_pubmed_des)
        for run in p.runs:
            run.font.size = Pt(11)
            run.font.name = 'Times New Roman'
        if prognosis_germline_des != 'nan':
            #print(prognosis_germline_des)
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run('Prognostic significance of germline mutation').bold=True
            for run in p.runs:
                run.font.size = Pt(12)
                run.font.name ='Times New Roman'

            if prognosis_germline_des == 'nan':
                prognosis_germline_des = 'ACMG recommendations are not relevant for this case'
            
            lines = prognosis_germline_des.split('\n')
            for line in lines:
                if line.startswith("*Note: Germline testing is recommended to determine if the identified pathogenic"):
                    p = document.add_paragraph()
                    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
                    p.paragraph_format.left_indent = Cm(2.5)
                    p.paragraph_format.right_indent = Cm(2.5)
                    p.add_run(line).bold = True
                    for run in p.runs:
                        run.font.size = Pt(11)
                        run.font.name = 'Times New Roman'
                else:
                    p = document.add_paragraph()
                    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
                    p.paragraph_format.left_indent = Cm(2.5)
                    p.paragraph_format.right_indent = Cm(2.5)
                    p.add_run(line)
                    for run in p.runs:
                        run.font.size = Pt(11)
                        run.font.name = 'Times New Roman'
                
    #Therapy
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Therapy' + '\n' + 'Approved therapy').bold = True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    if approved_therapy_des == 'nan':
        approved_therapy_des = 'Biomarker directed therapies could not be found in MyCancerGenome'
        p.add_run(approved_therapy_des)
        for run in p.runs:
            run.font.size = Pt(11)
            run.font.name = 'Times New Roman'
    else:
        lines = approved_therapy_des.split('\n')
        color_mapping = {
        'MediumPurple': RGBColor(147, 112, 219),
        'Pink': RGBColor(255, 20, 147),
        'Green': RGBColor(0, 128, 0),
        'Red': RGBColor(255, 0, 0),
        'Black': RGBColor(0, 0, 0)}
        current_color = None
        for line in lines:
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run(line)
            if line.startswith('Biomarker Criteria:'):
                current_color = color_mapping['Green']
            elif line.startswith('Predicted Response:'):
                current_color = color_mapping['Red']
            elif line.startswith('Clinical Setting(s):'):
                current_color = color_mapping['Black']
            elif line.startswith('Note:'):
                current_color = color_mapping['Black']
            elif line.startswith('Drug(s)'):
                current_color = color_mapping['MediumPurple']
            elif line.startswith('Cancer:'):
                current_color = color_mapping['Pink']
            for run in p.runs:
                run.font.size = Pt(11)
                run.font.name = 'Times New Roman'
                set_run_color(run, current_color)

    #In trial therapy
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.paragraph_format.space_after = Pt(0) 
    p.add_run('In trial therapy').bold = True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name = 'Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Therapies are being investigated in the clinical trials for patients of xxx or solid tumors with XXX mutation. The patient may be eligible to enroll in these clinical trials. Please see the “Clinical trial” section for more details.')
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman'
        
    #Drug resistance information
    drug_des=str(drug_des)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Drug resistance information').bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    if drug_des == 'nan':
        drug_des = 'Drug resistance information could not be found in the in-house database'
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run(drug_des)
        for run in p.runs:
            run.font.size = Pt(11)
            run.font.name ='Times New Roman'
    else:
        p.add_run('\n')
        for line in drug_des.split('\n'):
            if line.startswith('Drug:'):
                drug_name = line.split('Drug:')[1].strip()
                drug = drug_name.replace(" ","+")
                hyperlink_run = add_hyperlink(p, drug_name, f"https://www.google.com/search?q={drug}+mechanism+of+action")
                p.add_run(line.split(drug_name)[1])
            else:
                p = document.add_paragraph()
                p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
                p.paragraph_format.left_indent = Cm(2.5)
                p.paragraph_format.right_indent = Cm(2.5)
                p.add_run(line)
                for run in p.runs:
                    run.font.size = Pt(11)
                    run.font.name ='Times New Roman'
    p = document.add_paragraph()

In [22]:
ncbi_df = pd.read_excel('19K_Gene_GC_NCBI_Info.xlsx', sheet_name='All_Genes')
ncbi_df.drop(['GC_Name', 'GC_Aliases', 'Gene_ID', 'NCBI_Official_Symbol', 'NCBI_Full_Name', 'NCBI_Aliases'], axis=1, inplace=True)
ckbcore_df = pd.read_excel('Running Copy of Gene description_MyCancerGenome_CkBcore.xlsx')
ckbcore_df.rename(columns={'Gene name': 'Gene_Name'}, inplace=True)
prognosis_df = pd.read_excel('V2 Running Copy of Information related to prognosis.xlsx')
prognosis_df['Main Tumor type']=prognosis_df['Main Tumor type'].str.lower()
prognosis_df.rename(columns={'Gene': 'Gene_Name'}, inplace=True)
drug_res_df = pd.read_excel('V2 Running Copy of Drug resistance information.xlsx')
drug_res_df['Main Tumor type'] = drug_res_df['Main Tumor type'].str.lower()
drug_res_df.rename(columns={'Gene': 'Gene_Name'}, inplace=True)
tumor_types = [tumor.lower() for tumor in patient_info.get('Main Tumor Type', '').split(', ')]
biomarkers_df.columns = biomarkers_df.columns.str.strip()
data_biomarkers_df = biomarkers_df.drop(['Tier', 'Relevant Therapies (In this cancer type)', 'Relevant Therapies (In other cancer type)', 'Clinical Trials'], axis=1)
def extract_oncokb(text):
    if len(text.split()) == 2:
        return text
    else:
        text = text.replace(" ", "")
        match = re.search(r'p.\((.*?)\)', text)
        if match:
            return match.group(1)
        else:
            return None
data_biomarkers_df['oncokb'] = data_biomarkers_df['Genomic Alteration'].apply(extract_oncokb)
data_biomarkers_df.rename(columns={'Gene': 'Gene_Name'}, inplace=True)
data_biomarkers_df.drop(['Genomic Alteration'], axis=1, inplace=True)
variants_description = data.get("Variants Description", [])
genes_involved=[]
if snv_df is not None:
    snv_df.columns = snv_df.columns.str.strip()
    snv_df['Gene_Name'] = snv_df.apply(lambda row: f"{row['Gene'].split()[0]}", axis=1)
    snv_df = pd.merge(snv_df, data_biomarkers_df, on='Gene_Name', how='left')
    snv_df['Gene_Heading'] = snv_df.apply(lambda row: f"{row['Gene'].split()[0]}\n{row['Genomic alteration']}", axis=1)
    snv_df.rename(columns={'Total Coverage/VAF ': 'Variant Allelic Frequency (VAF%)'}, inplace=True)
    snv_df['Variant Allelic Frequency (VAF%)'] = snv_df['Variant Allelic Frequency (VAF%)'].str.replace(r'\d+X\s*', '', regex=True)
    snv_df = pd.merge(snv_df, ncbi_df, on='Gene_Name', how='left')
    snv_df = pd.merge(snv_df, ckbcore_df, on='Gene_Name', how='left')
    prognosis_filtered_tumor_types = prognosis_df[(prognosis_df['Gene_Name'].isin(snv_df['Gene_Name']))&(prognosis_df['Main Tumor type'].isin(tumor_types))]
    prognosis_filtered_solid_tumors = prognosis_df[(prognosis_df['Gene_Name'].isin(snv_df['Gene_Name']))&(prognosis_df['Main Tumor type'] == 'solid tumours')]
    prognosis_filtered_df = pd.concat([prognosis_filtered_tumor_types, prognosis_filtered_solid_tumors])
    prognosis_filtered_df =prognosis_filtered_df.groupby('Gene_Name')['Prognostic significance'].apply(lambda x: '\n'.join(x)).reset_index()
    snv_df = pd.merge(snv_df, prognosis_filtered_df, on='Gene_Name', how='left')
    drug_res_filtered_tumor_types = drug_res_df[(drug_res_df['Gene_Name'].isin(snv_df['Gene_Name']))&(drug_res_df['Main Tumor type'].isin(tumor_types))]
    drug_res_filtered_solid_tumors = drug_res_df[(drug_res_df['Gene_Name'].isin(snv_df['Gene_Name']))&(drug_res_df['Main Tumor type'] == 'solid tumours')]
    drug_res_filtered_df = pd.concat([drug_res_filtered_tumor_types, drug_res_filtered_solid_tumors])
    drug_res_filtered_df["Drug_name_description"] = 'Drug: '+ drug_res_filtered_df["Potential drug \nresistance"].fillna('') + '\n' + drug_res_filtered_df["Description"].fillna('')+ '\n' 
    drug_res_filtered_df = drug_res_filtered_df.groupby('Gene_Name')['Drug_name_description'].apply(lambda x: '\n'.join(x)).reset_index()
    snv_df = pd.merge(snv_df, drug_res_filtered_df, on='Gene_Name', how='left')
    snv_df.drop_duplicates(inplace=True)
    for index, row in snv_df.iterrows():
        gene_name = row['Gene_Heading']
        formatted_gene_name = gene_name.replace("\n", "").replace(" - ", "-").replace("-", " - ")
        formatted_gene_name = formatted_gene_name.replace("c.", ":c.").replace("p.", ":p.")
        formatted_gene_name = formatted_gene_name.replace(" ", "")
        formatted_gene_name = formatted_gene_name.replace(" -  - ", " - ")
        formatted_gene_name = formatted_gene_name.replace("--", " - ")
        formatted_gene_name = formatted_gene_name.replace("-:", "-")
        genes_involved.append(formatted_gene_name)

if cnv_df is not None:
    cnv_df.columns = cnv_df.columns.str.strip()
    cnv_df.rename(columns={'Gene': 'Gene_Name'}, inplace=True)
    cnv_df['Gene_Name'] = cnv_df['Gene_Name'].str.replace(' ', '')
    cnv_df = pd.merge(cnv_df, data_biomarkers_df, on='Gene_Name', how='left')
    cnv_df['Gene_Heading'] = cnv_df.apply(lambda row: f"{row['Gene_Name']}\n{row['Locus'].split()[1]}x{row['Copy Number']}\n{row['CNV type']}", axis=1)
    cnv_df['Gene_CNV']= cnv_df.apply(lambda row: f"{row['Gene_Name']} {row['CNV type']}", axis=1)
    cnv_df = pd.merge(cnv_df, ncbi_df, on='Gene_Name', how='left')
    cnv_df = pd.merge(cnv_df, ckbcore_df, on='Gene_Name', how='left')
    prognosis_filtered_tumor_types = prognosis_df[(prognosis_df['Gene_Name'].isin(cnv_df['Gene_Name']))&(prognosis_df['Main Tumor type'].isin(tumor_types))]
    prognosis_filtered_solid_tumors = prognosis_df[(prognosis_df['Gene_Name'].isin(cnv_df['Gene_Name']))&(prognosis_df['Main Tumor type'] == 'solid tumours')]
    prognosis_filtered_df = pd.concat([prognosis_filtered_tumor_types, prognosis_filtered_solid_tumors])
    prognosis_filtered_df =prognosis_filtered_df.groupby('Gene_Name')['Prognostic significance'].apply(lambda x: '\n'.join(x)).reset_index()
    cnv_df = pd.merge(cnv_df, prognosis_filtered_df, on='Gene_Name', how='outer')
    drug_res_filtered_tumor_types = drug_res_df[(drug_res_df['Gene_Name'].isin(cnv_df['Gene_Name']))&(drug_res_df['Main Tumor type'].isin(tumor_types))]
    drug_res_filtered_solid_tumors = drug_res_df[(drug_res_df['Gene_Name'].isin(cnv_df['Gene_Name']))&(drug_res_df['Main Tumor type'] == 'solid tumours')]
    drug_res_filtered_df = pd.concat([drug_res_filtered_tumor_types, drug_res_filtered_solid_tumors])
    drug_res_filtered_df["Drug_name_description"] = 'Drug: '+ drug_res_filtered_df["Potential drug \nresistance"].fillna('') + '\n' + drug_res_filtered_df["Description"].fillna('')+ '\n' 
    drug_res_filtered_df = drug_res_filtered_df.groupby('Gene_Name')['Drug_name_description'].apply(lambda x: '\n'.join(x)).reset_index()
    cnv_df= pd.merge(cnv_df, drug_res_filtered_df, on='Gene_Name', how='outer')
    for index, row in cnv_df.iterrows():
        gene_name = row['Gene_CNV']
        genes_involved.append(gene_name)

if fusion_df is not None:
    for gene_pair in fusion_df['Gene'].unique():
        formatted_gene_name = gene_pair.replace("\n", "").replace(" - ", "-").replace("-", " - ")
        formatted_gene_name = formatted_gene_name.replace("c.", ":c.").replace("p.", ":p.")
        formatted_gene_name = formatted_gene_name.replace(" ", "")
        formatted_gene_name = formatted_gene_name.replace(" -  - ", " - ")
        formatted_gene_name = formatted_gene_name.replace("--", " - ")
        formatted_gene_name = formatted_gene_name.replace("-:", "-")
        genes_involved.append(formatted_gene_name)


def gene_varient_description(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(9.81)
    new_height = Cm(0.56)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Gene_variant_des.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    prognosis_des='nan'
    approved_therapy_des = 'nan'
    prognosis_germline_des = 'nan'
    drug_des = 'nan'
    if not snv_df.empty:
        for index, row in snv_df.iterrows():
            dna_gene_heading = row['Gene_Heading']
            gene =row['Gene_Name']
            ncbi_summary = row['NCBI_Summary']
            gene_roles = row.get('Gene Role', None)
            if gene_roles is not None and pd.notna(gene_roles):
                gene_roles = gene_roles.split(' (')[0].replace('Both: ', '')
                gene_roles = re.sub(r'\([^)]*\)', '', gene_roles)
            else:
                gene_roles = 'There is no Gene role information for this gene'
            gene_ckb_des = row['Gene Description                                                                                                                                                                                        [Source : CKB core  https://ckb.jax.org/gene/grid]']
            df=pd.DataFrame()
            if ' ' in gene.strip():
                gene=gene.lower()
                gene=gene.replace(' ', '-')
                df=pd.concat([df,For_Alterations(gene)])
            elif ' ' not in gene.lower().strip():
                df=pd.concat([df,For_Genes(gene)])
            my_cancer_des = df['Gene_Role'].iloc[0].replace('\n', '').replace('  ', '') if not df.empty else 'There is no My cancer genome information for this gene'
            condition = (~df['Drugs'].eq('')) & (~df['Cancer'].eq('')) & (~df['Biomarker_Criteria'].eq(''))
            df['my_cancer_data'] = ''
            df.loc[condition, 'my_cancer_data'] = 'Drug(s): ' + df.loc[condition, 'Drugs'] + '\n' + 'Cancer: ' + df.loc[condition, 'Cancer'] + '\n' + df.loc[condition, 'Biomarker_Criteria']
            df = df[df['Gene_Name'] == gene]
            approved_therapy_des = '\n'.join(df['my_cancer_data']) if not df.empty and 'my_cancer_data' in df else 'Biomarker directed therapies could not be found in MyCancerGenome'
            if pd.isna(approved_therapy_des) or approved_therapy_des == '':
                approved_therapy_des = 'Biomarker directed therapies could not be found in MyCancerGenome' 
            prognosis_des=row['Prognostic significance']
            variant_classification = row['Clinical Significance']
            vaf_percentage = row['Variant Allelic Frequency (VAF%)']
            vaf_percentage = vaf_percentage.replace('%', '').replace(',', '.').strip()
            oncokb = row['oncokb']
            if 'Pathogenic' in variant_classification or 'Likely Pathogenic' in variant_classification and float(vaf_percentage) >= 20:
                medgen_df = ACMG_dataextraction(gene)
                if medgen_df is not None:
                    if not medgen_df.empty:
                        matching_rows = medgen_df[medgen_df['Gene_Name'] == gene]
                        if not matching_rows.empty and 'disease_characteristics' in matching_rows:
                            prognosis_germline_des = ' '.join(matching_rows['disease_characteristics'])
                            prognosis_germline_des += f"\n*Note: Germline testing is recommended to determine if the identified pathogenic {gene} mutation is germline or somatic."
                    else:
                        prognosis_germline_des = 'ACMG recommendations are not relevant for this case' 
            elif 'Variant of Uncertain Significance' in variant_classification and float(vaf_percentage) >= 20:
                medgen_df = ACMG_dataextraction(gene)
                if medgen_df is not None:
                    if not medgen_df.empty:
                        matching_rows = medgen_df[medgen_df['Gene_Name'] == gene]
                        if not matching_rows.empty and 'disease_characteristics' in matching_rows:
                            prognosis_germline_des = ' '.join(matching_rows['disease_characteristics'])
                            prognosis_germline_des += f"\n*This patient has a VUS-Deleterious mutation in {gene} with {vaf_percentage}% variant allele frequency."
                    else:
                        prognosis_germline_des = 'ACMG recommendations are not relevant for this case'
        
            drug_des=row['Drug_name_description']
            add_gene_summary_text(document, dna_gene_heading, ncbi_summary, gene_roles, gene_ckb_des, my_cancer_des, '', oncokb, prognosis_des, prognosis_germline_des, approved_therapy_des, drug_des)
    print('Done with Dna sequence varaints')
    
    if not cnv_df.empty:    
        for index, row in cnv_df.iterrows():
            copy_gene_heading = row['Gene_Heading']
            variant_class = row['CNV type'].lower()
            if variant_class == 'deletion':
                gene= row['Gene_Name'] + '-Loss'
            else:
                gene= row['Gene_Name'] + '-'+row['CNV type']
            ncbi_summary = row['NCBI_Summary']
            gene_roles = row.get('Gene Role', None)
            if gene_roles is not None and pd.notna(gene_roles):
                gene_roles = gene_roles.split(' (')[0].replace('Both: ', '')
                gene_roles = re.sub(r'\([^)]*\)', '', gene_roles)
            else:
                gene_roles = 'There is no Gene role information for this gene'
            gene_ckb_des = row['Gene Description                                                                                                                                                                                        [Source : CKB core  https://ckb.jax.org/gene/grid]']
            df=pd.DataFrame()
            gene=gene.lower()
            gene=gene.replace(' ', '-')
            df=pd.concat([df,For_Alterations(gene)])
            my_cancer_des = df['Gene_Role'].iloc[0].replace('\n', '').replace('  ', '') if not df.empty else 'There is no My cancer genome information for this gene'
            condition = (~df['Drugs'].eq('')) & (~df['Cancer'].eq('')) & (~df['Biomarker_Criteria'].eq(''))
            df['my_cancer_data'] = ''
            df.loc[condition, 'my_cancer_data'] = 'Drug(s): ' + df.loc[condition, 'Drugs'] + '\n' + 'Cancer: ' + df.loc[condition, 'Cancer'] + '\n' + df.loc[condition, 'Biomarker_Criteria']
            df = df[df['Gene_Name'] == gene]
            approved_therapy_des = '\n'.join(df['my_cancer_data']) if not df.empty and 'my_cancer_data' in df else 'Biomarker directed therapies could not be found in MyCancerGenome'
            if pd.isna(approved_therapy_des) or approved_therapy_des == '':
                approved_therapy_des = 'Biomarker directed therapies could not be found in MyCancerGenome'
            oncokb = row['oncokb']
            prognosis_des=row['Prognostic significance']
            drug_des=row['Drug_name_description']
            add_gene_summary_text(document, copy_gene_heading, ncbi_summary, gene_roles, gene_ckb_des, my_cancer_des, '', oncokb, prognosis_des, 'ACMG recommendations are not relevant for this case', approved_therapy_des, drug_des)
    print('Done with copy number')
    '''
    if fusion_variants_data:
        try:
            if not gene_fusions_df.empty:
                for index, row in gene_fusions_df.iterrows():
                    fusion_heading = row['Gene_Heading']
                    altered_var = row['altered_variant_para']
                    gene = fusion_heading 
                    df=pd.DataFrame()
                    gene=gene.lower()
                    gene=gene.replace(' ', '-')
                    df=pd.concat([df,For_Alterations(gene)])
                    my_cancer_des = df['Gene_Role'].iloc[0].replace('\n', '').replace('  ', '') if not df.empty else 'There is no My cancer genome information for this gene'
                    condition = (~df['Drugs'].eq('')) & (~df['Cancer'].eq('')) & (~df['Biomarker_Criteria'].eq(''))
                    df['my_cancer_data'] = ''
                    df.loc[condition, 'my_cancer_data'] = 'Drug(s): ' + df.loc[condition, 'Drugs'] + '\n' + 'Cancer: ' + df.loc[condition, 'Cancer'] + '\n' + df.loc[condition, 'Biomarker_Criteria']
                    df = df[df['Gene_Name'] == gene]
                    approved_therapy_des = '\n'.join(df['my_cancer_data']) if not df.empty and 'my_cancer_data' in df else 'Biomarker directed therapies could not be found in MyCancerGenome'
                    if pd.isna(approved_therapy_des) or approved_therapy_des == '':
                        approved_therapy_des = 'Biomarker directed therapies could not be found in MyCancerGenome'
                    ncbi_summary_accumulator = ''
                    gene_roles_accumulator = ''
                    gene_ckb_des_accumulator = ''
                    prognosis_des_accumulator = ''
                    drug_des_accumulator = ''
                    
                    for index, row in fusion_df.iterrows():
                        gene = row['Gene']
                        if not pd.isna(row['NCBI_Summary']):
                            ncbi_summary_accumulator += gene + '\n' + row['NCBI_Summary'] + '\n'
                        else:
                            ncbi_summary_accumulator += gene + '\n' + 'NCBI Summary information could not be found in the in-house database' + '\n'
    
                        if not pd.isna(row['Gene Role']):
                            gene_roles = row.get('Gene Role', None)
                            if gene_roles is not None and pd.notna(gene_roles):
                                gene_roles = gene_roles.split(' (')[0].replace('Both: ', '')
                                gene_roles = re.sub(r'\([^)]*\)', '', gene_roles)
                                gene_roles_accumulator += gene_roles + '\n'
                        else:
                            gene_roles_accumulator += 'There is no Gene role information for this gene' + '\n'
    
                        gene_ckb_des_accumulator += gene + '\n' + row['Gene Description                                                                                                                                                                                        [Source : CKB core  https://ckb.jax.org/gene/grid]'] + '\n'
    
                        if not pd.isna(row['Prognostic significance']):
                            prognosis_des_accumulator += gene + '\n' + row['Prognostic significance'] + '\n'
                        else:
                            prognosis_des_accumulator += gene + '\n' + 'Prognostic significance information could not be found in the in-house database' + '\n'
    
                        if not pd.isna(row['Drug_name_description']):
                            drug_des_accumulator += gene + '\n' + row['Drug_name_description'] + '\n'
                        else:
                            drug_des_accumulator += gene + '\n' + 'Drug resistance information could not be found in the in-house database' + '\n'
    
                    add_gene_summary_text(document, fusion_heading, ncbi_summary_accumulator, gene_roles_accumulator, gene_ckb_des_accumulator, my_cancer_des, altered_var, '', prognosis_des_accumulator, 'ACMG recommendations are not relevant for this case', approved_therapy_des, drug_des_accumulator)
    
        except Exception as e:
            print(f"An error occurred: {e}")
        '''

In [23]:
def hrr_table(document, data):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(1.4)
        section.right_margin = Cm(2)

    table = document.add_table(rows=len(data) + 1, cols=2)  # Add 1 for header row
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(8.25)
    table.columns[1].width = Cm(8.25)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.style = 'Table Grid'
    table.style.font.name = 'Times New Roman'
    table.style.font.size = Pt(11)

    header_row = table.rows[0].cells
    header_row[0].text = "Gene/Genomic Alteration"
    header_row[1].text = "Findings"
    
    for i, (index, row_data) in enumerate(data.iterrows()):
        row_cells = table.rows[i + 1].cells  # Add 1 to skip header row
        row_cells[0].text = row_data["Gene/Genomic Alteration "]
        row_cells[1].text = row_data["Finding "]
        
    for cell in table.rows[0].cells:
        cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell.paragraphs[0].runs[0].font.bold = True
        cell.paragraphs[0].runs[0].font.size = Pt(11)
        
    return table

In [24]:
def hrr_genes(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(12.17)
    new_height = Cm(0.85)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'HRR_genes.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    loh_score = hrr_df.loc[hrr_df['Gene/Genomic Alteration '] == 'LOH percentage', 'Finding '].str.extract(r'(\d+\.\d+)').iloc[0, 0] if not hrr_df.loc[hrr_df['Gene/Genomic Alteration '] == 'LOH percentage', 'Finding '].empty else None
    loh_score_str = str(loh_score) + '%' if loh_score is not None else None
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Homologous recombination repair (HRR) pathway genes play a vital role in maintaining genome stability and tumor suppression. Alterations in HRR genes can lead to genome instability which in turn may increase the risk of developing tumors. Thus knowing the alterations in HRR genes can act as potential biomarkers to decide the personalized therapy to be undertaken.')
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    
    if loh_score_str and loh_score_str in ['0.0%', '0%']:
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run('HRR Genes are not detected and Not applicable for the case')
        for run in p.runs:
            run.font.size = Pt(12)
            run.font.name = 'Times New Roman'
    else:
        note_text = f'Note: This patient shows {loh_score_str} of LOH within the HRR gene panel that includes BRCA1/2. To further confirm the HRR/HRD status, assessment of HRR/HRD using orthogonal methods is recommended.'
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        for char in note_text:
            run = p.add_run(char)
            run.font.size = Pt(11)
            run.font.name = 'Times New Roman'
            if loh_score_str is not None and char in loh_score_str:
                run.bold = True

        hrr_table(document, hrr_df)
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run('*Note: From the total of 46 HRR genes (including BRCA1 and BRCA2) covered in Oncomine Comprehensive Assay Plus, these alterations are Identified. Confirmation with other orthogonal methods is recommended.')
        for run in p.runs:
            run.font.size = Pt(10)
            run.font.name ='Times New Roman'
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run('*Note: Homologous recombination repair (HRR) genes were defined from published evidence in relevant therapies, clinical guidelines, as well as clinical trials, and include - BRCA1, BRCA2, ATM, BARD1, BLM, BRIP1, CDK12, CHEK1, CHEK2, FANCL, NBN, PALB2, POLD1, POLE, PPP2R2A, RAD51B, RAD51C, RAD51D, and RAD54L.')     
        for run in p.runs:
            run.font.size = Pt(10)
            run.font.name ='Times New Roman'


In [25]:
def clinical_trial_table(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(2)
        section.right_margin = Cm(2)
        
    line1_paragraph = document.add_paragraph("Following ongoing clinical trials are relevant for the patient's diagnosis and genomic profile")
    line1_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    line1_paragraph.paragraph_format.left_indent = Cm(2.5)
    line1_paragraph.paragraph_format.right_indent = Cm(2.5)
    line1_paragraph.paragraph_format.space_after = Cm(0)
    for run in line1_paragraph.runs:
        run.font.size = Pt(13)
        run.font.name = 'Times New Roman'

    line2_paragraph = document.add_paragraph("(Additional trials that are investigating immunotherapies relevant for the patient’s diagnosis or relevant trials that are recruiting patients in India are also listed below)")
    line2_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    line2_paragraph.paragraph_format.left_indent = Cm(2.5)
    line2_paragraph.paragraph_format.right_indent = Cm(2.5)
    line2_paragraph.paragraph_format.space_after = Cm(0)
    for run in line2_paragraph.runs:
        run.font.size = Pt(12)
        run.font.name = 'Times New Roman'
    
    p = document.add_paragraph()
    table = document.add_table(rows=16, cols=6)  
    table.style = 'Table Grid'
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.autofit = False
    table.allow_autofit = False
    widths = [3.5, 5, 3, 1.85, 3, 2.90]  
    for i, width in enumerate(widths):
        table.cell(0, i).width = Cm(width)
        
    header = table.rows[0].cells
    header[0].text = "Relevant Variant/ Parameter"
    header[1].text = "Clinical trial"
    header[2].text = "Intervention"
    header[3].text = "Phase"
    header[4].text = "Trial identifier"
    header[5].text = "Recruiting location"

    for cell in header:
        cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        cell.paragraphs[0].runs[0].font.size = Pt(12)
        cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell.paragraphs[0].runs[0].bold = True

    for row in table.rows[1:]:
        for cell in row.cells:
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.font.size = Pt(10)
                    run.font.name = 'Times New Roman'

    return table

In [26]:
def clinical_trial(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(6.46)
    new_height = Cm(0.97)
    paragraph = document.add_paragraph()
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Clinical_trials.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    paragraph = document.add_paragraph()
    clinical_trial_table(document)
    p = document.add_paragraph()
    document.paragraphs[-1].paragraph_format.space_before = Pt(0)
    note1 = document.add_paragraph("Note 1: Confirmation of somatic/germline status of certain variants will be necessary to determine the patient's eligibility for the listed trials.")
    note1.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note1.paragraph_format.left_indent = Cm(2.5)
    note1.paragraph_format.right_indent = Cm(2.5)
    note1.paragraph_format.space_after = Cm(0)
    note1.paragraph_format.line_spacing = 1
    for run in note1.runs:
        run.font.size = Pt(10)
        run.font.name = 'Times New Roman'
    note2 = document.add_paragraph("Note 2: Patient’s known treatment history, as per the discharge summary is also taken into consideration while matching these trials.")
    note2.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note2.paragraph_format.left_indent = Cm(2.5)
    note2.paragraph_format.right_indent = Cm(2.5)
    note1.paragraph_format.space_after = Cm(0)
    for run in note2.runs:
        run.font.size = Pt(10)
        run.font.name = 'Times New Roman'
    note3 = document.add_paragraph("Note 3:To further determine the patient's eligibility for the trial, careful matching of the inclusion criteria based upon the patient’s condition and other clinical parameters is necessary.")
    note3.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note3.paragraph_format.left_indent = Cm(2.5)
    note3.paragraph_format.right_indent = Cm(2.5)
    note2.paragraph_format.space_after = Cm(0)
    for run in note3.runs:
        run.font.size = Pt(10)
        run.font.name = 'Times New Roman'

In [27]:
def quality_control(document, data):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(2)
        section.right_margin = Cm(2)

    table = document.add_table(rows=2, cols=2)
    table.style = 'Table Grid'
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(4.00)
    table.columns[1].width = Cm(4.00)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.style.font.name = 'Times New Roman'
    table.style.font.size = Pt(11)
    mean_depth_coverage = re.search(r'Average base coverage depth of (\d+\,\d+)', data).group(1)
    target_base_coverage_500x = re.search(r'Target base coverage at 500X is (\d+\.\d+)%', data).group(1)
    table.cell(0, 0).text = 'Mean depth coverage'
    table.cell(0, 1).text = mean_depth_coverage
    table.cell(1, 0).text = 'Target Base coverage at 500X'
    table.cell(1, 1).text = f'{target_base_coverage_500x}%'
    return table

In [28]:
def supplementary_table(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(2)
        section.right_margin = Cm(2)
    table = doc.add_table(rows=11, cols=2)
    table.style = 'Table Grid'
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(3.31)
    table.columns[1].width = Cm(13.36)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.cell(0, 0).merge(table.cell(0, 1)).text = 'Variant Classification'
    table.cell(0, 0).paragraphs[0].runs[0].bold = True
    table.cell(1, 0).merge(table.cell(1, 1)).text = 'Tier I: Variants of Strong clinical significance'
    table.cell(1, 0).paragraphs[0].runs[0].bold = True
    table.cell(2, 0).text = 'Level A'
    table.cell(2, 0).paragraphs[0].runs[0].bold = True
    table.cell(2, 1).text = 'FDA-approved therapy included in professional guidelines'
    table.cell(3, 0).text = 'Level B'
    table.cell(3, 0).paragraphs[0].runs[0].bold = True
    table.cell(3, 1).text = 'Well-powered studies with consensus from experts in the field'
    table.cell(4, 0).merge(table.cell(4, 1)).text = 'Tier II: Variants of Potential Clinical Significance'
    table.cell(4, 0).paragraphs[0].runs[0].bold = True
    table.cell(5, 0).text = 'Level C'
    table.cell(5, 0).paragraphs[0].runs[0].bold = True
    table.cell(5, 1).text = 'FDA-approved therapies for different tumor types or investigational therapies\nMultiple small published studies with some consensus'
    table.cell(6, 0).text = 'Level D'
    table.cell(6, 0).paragraphs[0].runs[0].bold = True
    table.cell(6, 1).text = 'Preclinical trials or a few case reports without consensus'
    table.cell(7, 0).merge(table.cell(7, 1)).text = 'Tier III: Variants of Unknown Clinical Significance'
    table.cell(7, 0).paragraphs[0].runs[0].bold = True
    table.cell(8, 0).merge(table.cell(8, 1)).text = 'Not observed at a significant allele frequency in the general or specific subpopulation databases, or pan-cancer or tumor-specific variant databases. No convincing published evidence of cancer association'
    table.cell(9, 0).merge(table.cell(9, 1)).text = 'Tier IV: Benign or Likely Benign Variants'
    table.cell(9, 0).paragraphs[0].runs[0].bold = True
    table.cell(10, 0).merge(table.cell(10, 1)).text = 'Observed at a significant allele frequency in the general or specific subpopulation databases. No existing published evidence of cancer association'
    for row in table.rows:
        for cell in row.cells:
            for paragraph in cell.paragraphs:
                if paragraph.runs:
                    run = paragraph.runs[0]
                    run.font.name = 'Times New Roman'
                    run.font.size = Pt(10)

In [29]:
def supplementary_information(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    p = document.add_paragraph()
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Supplementary Information').bold=True
    for run in p.runs:
        run.font.size = Pt(14)
        run.font.name ='Times New Roman'
        run.font.underline = True
    p = document.add_paragraph()
    new_width = Cm(7.41)
    new_height = Cm(0.65)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Test_des.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Comprehensive genomic profiling (CGP) is advancing precision oncology research through the analysis of multiple relevant biomarkers in a single next-generation sequencing (NGS) test. The test can detect all types of single-gene variants, such as single-nucleotide variants (SNVs), insertions and deletions (indels), novel and known fusions,  splice variants, and copy number variants (CNVs), including both copy number gains and losses.A  study potential responses to immunotherapies with measurement of tumor mutational burden (TMB) and predisposition to genetic hypermutability by comparing microsatellite instability (MSI) regions, and analyse mutational signatures for insights into etiological factors in tumorigenesis. It can detect both gene-level and sample-level loss of heterozygosity (LOH) to assess genomic instability and mutations in 46 key genes in the homologous  recombination repair (HRR) pathway.')
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman' 
        
    p = document.add_paragraph()
    new_width = Cm(7.41)
    new_height = Cm(0.71)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Quality_Control_Statics.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    sentence = data.get("QC statistics", " ")
    quality_control(document, sentence)
    
    p = document.add_paragraph()
    new_width = Cm(6.49)
    new_height = Cm(0.66)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Genes_analyzed.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    header_run = p.add_run('Genes Assayed for the Detection of DNA Sequence Variants: ')
    header_run.bold = True
    header_run.underline = True
    header_run.font.name = 'Times New Roman'
    header_run.font.size = Pt(9)
    genes_text = ('ABL1, ABL2, ACVR1, ACVR2A, AKT1, AKT2, AKT3, ALK, AMER1, APC, AR, ARAF, ARID1A, ARID1B, ARID2, ASXL1, ASXL2, ATM, ATP1A1, ATR, ATRX, AURKA, AURKC, AXIN1, AXIN2, AXL, B2M, BAP1, BCL2, BCL2L12, BCL6, BCOR, BCR, BLM, BMP5, BRAF, BRCA1, BRCA2, BRIP1, BTK, CACNA1D, CALR, CARD11, CASP8, CBL, CCND1, CCND2, CCND3, CCNE1, CD79B, CDC73, CDH1, CDK4, CDK6, CDKN2A, CDKN2C, CHD4, CHEK2, CIC, CREBBP, CSF1R, CTCF, CTNNB1, CUL1, CUL3, CYP2D6, CYSLTR2, DDR2, DDX3X, DGCR8, DICER1, DNMT3A, DPYD, DROSHA, E2F1, EGFR, EIF1AX, EP300, EPAS1, EPHA2, ERBB2, ERBB3, ERBB4, ERCC2, ERCC5, ERRFI1, ESR1, EZH2, FAM135B, FANCM, FBXW7, FGF7, FGFR1, FGFR2, FGFR3, FGFR4, FLT3, FLT4, FOXA1, FOXL2, FOXO1, FUBP1, GATA2, GATA3, GLI1, GNA11, GNA13, GNAQ, GNAS, GPS2, H2BC5, H3-3A, H3-3B, H3C2, HIF1A, HNF1A, HRAS, ID3, IDH1, IDH2, IKBKB, IL6ST, IL7R, IRF4, IRS4, JAK1, JAK2, JAK3, KDM6A, KDR, KEAP1, KIT, KLF4, KLF5, KMT2B, KMT2D, KNSTRN, KRAS, LARP4B, LATS1, MAGOH, MAP2K1, MAP2K2, MAP2K4, MAP2K7, MAP3K4, MAPK1, MAPK8, MAX, MDM4, MECOM, MED12, MEF2B, MEN1, MET, MGA, MITF, MLH3, MPL, MSH3, MSH6, MTOR, MYC, MYCN, MYD88, MYOD1, NBN, NCOR1, NF1, NF2, NFE2L2, NOTCH1, NOTCH2, NRAS, NSD2, NT5C2, NTRK1, NTRK2, NTRK3, NUP93, PARP1, PAX5, PBRM1, PCBP1, PDGFRA, PDGFRB, PHF6, PIK3C2B, PIK3CA, PIK3CB, PIK3CD, PIK3CG, PIK3R1, PIK3R2, PIM1, PLCG1, PMS2, POLE, PPM1D, PPP2R1A, PPP6C, PRKACA, PTCH1, PTEN, PTPN11, PTPRD, PXDNL, RAC1, RAD50, RAD51, RAF1, RARA, RB1, RET, RGS7, RHEB, RHOA, RICTOR, RIT1, RNF43, ROS1, RPL10, RPL5, RUNX1, RUNX1T1, SDHD, SETBP1, SETD2, SF3B1, SIX1, SIX2, SLCO1B3, SLX4, SMAD2, SMAD4, SMARCA4, SMARCB1, SMC1A, SMO, SNCAIP, SOCS1, SOS1, SOX2, SPOP, SRC, SRSF2, STAG2, STAT3, STAT5B, STAT6, STK11, TAF1, TCF7L2, TERT, TET2, TGFBR1, TGFBR2, TNFAIP3, TOP1, TP53, TPMT, TRRAP, TSC2, TSHR, U2AF1, UGT1A1, USP8, VHL, WAS, WT1, XPO1, XRCC2, ZFHX3, ZNF217, ZNF429')
    genes_run = p.add_run(genes_text)
    genes_run.italic = True
    genes_run.font.name = 'Times New Roman'
    genes_run.font.size = Pt(9)   
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    header_run = p.add_run('Genes Assayed for the Detection of Copy Number Variations: ')
    header_run.bold = True
    header_run.underline = True
    header_run.font.name = 'Times New Roman'
    header_run.font.size = Pt(9)
    genes_text = ('ABCB1, ABL1, ABL2, ABRAXAS1, ACVR1B, ACVR2A, ADAMTS12, ADAMTS2, AKT1, AKT2, AKT3, ALK, AMER1, APC, AR, ARAF, ARHGAP35, ARID1A, ARID1B, ARID2, ARID5B, ASXL1, ASXL2, ATM, ATR, ATRX, AURKA, AURKC, AXIN1, AXIN2, AXL, B2M, BAP1, BARD1, BCL2, BCL2L12, BCL6, BCOR, BLM, BMPR2, BRAF, BRCA1, BRCA2, BRIP1, CARD11, CASP8, CBFB, CBL, CCND1, CCND2, CCND3, CCNE1, CD274, CD276, CDC73, CDH1, CDH10, CDK12, CDK4, CDK6, CDKN1A, CDKN1B, CDKN2A, CDKN2B, CDKN2C, CHD4, CHEK1, CHEK2, CIC, CREBBP, CSMD3, CTCF, CTLA4, CTNND2, CUL3, CUL4A, CUL4B, CYLD, CYP2C9, DAXX, DDR1, DDR2, DDX3X, DICER1, DNMT3A, DOCK3, DPYD, DSC1, DSC3, EGFR, EIF1AX, ELF3, EMSY, ENO1, EP300, EPCAM, EPHA2, ERAP1, ERAP2, ERBB2, ERBB3, ERBB4, ERCC2, ERCC4, ERRFI1, ESR1, ETV6, EZH2, FAM135B, FANCA, FANCC, FANCD2, FANCE, FANCF, FANCG, FANCI,  FANCL, FANCM, FAT1, FBXW7, FGF19, FGF23, FGF3, FGF4, FGF9, FGFR1, FGFR2, FGFR3, FGFR4, FLT3, FLT4, FOXA1, FUBP1, FYN, GATA2, GATA3, GLI3, GNA13, GNAS, GPS2, H3-3A, H3-3B, HDAC2, HDAC9, HLA-A, HLA-B, HNF1A, IDH2, IGF1R, IKBKB, IL7R, INPP4B, JAK1, JAK2, JAK3, KDM5C, KDM6A, KDR, KEAP1, KIT, KLF5, KMT2A, KMT2B, KMT2C, KMT2D, KRAS, LARP4B, LATS1, LATS2, MAGOH, MAP2K1, MAP2K4, MAP2K7, MAP3K1, MAP3K4, MAPK1, MAPK8, MAX, MCL1, MDM2, MDM4, MECOM, MEF2B, MEN1, MET, MGA, MITF, MLH1, MLH3, MPL, MRE11, MSH2, MSH3, MSH6, MTAP, MTOR, MUTYH, MYC, MYCL, MYCN, MYD88, NBN, NCOR1, NF1, NF2, NFE2L2, NOTCH1, NOTCH2, NOTCH3, NOTCH4, NRAS, NTRK1, NTRK3, PALB2, PARP1, PARP2, PARP3, PARP4, PBRM1, PCBP1, PDCD1, PDCD1LG2, PDGFRA, PDGFRB, PDIA3, PGD, PHF6, PIK3C2B, PIK3CA, PIK3CB, PIK3R1, PIK3R2, PIM1, PLCG1, PMS1, PMS2, POLD1, POLE, POT1, PPM1D, PPP2R1A, PPP2R2A, PPP6C, PRDM1, PRDM9, PRKACA, PRKAR1A, PTCH1, PTEN, PTPN11, PTPRT, PXDNL, RAC1, RAD50, RAD51, RAD51B, RAD51C, RAD51D, RAD52, RAD54L, RAF1, RARA, RASA1, RASA2, RB1, RBM10, RECQL4, RET, RHEB, RICTOR, RIT1, RNASEH2A, RNASEH2B, RNF43, ROS1, RPA1, RPS6KB1, RPTOR, RUNX1, SDHA, SDHB, SDHD, SETBP1, SETD2, SF3B1, SLCO1B3, SLX4, SMAD2, SMAD4, SMARCA4, SMARCB1, SMC1A, SMO, SOX9, SPEN, SPOP, SRC, STAG2, STAT3, STAT6, STK11, SUFU, TAP1, TAP2, TBX3, TCF7L2, TERT, TET2, TGFBR2, TNFAIP3, TNFRSF14, TOP1, TP53, TP63, TPMT, TPP2, TSC1, TSC2, U2AF1, USP8, USP9X, VHL, WT1, XPO1, XRCC2, XRCC3, YAP1, YES1, ZFHX3, ZMYM3, ZNF217, ZNF429, ZRSR2')    
    genes_run = p.add_run(genes_text)
    genes_run.italic = True
    genes_run.font.name = 'Times New Roman'
    genes_run.font.size = Pt(9)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    header_run = p.add_run('Genes Assayed for the Detection of Fusions: ')
    header_run.bold = True
    header_run.underline = True
    header_run.font.name = 'Times New Roman'
    header_run.font.size = Pt(9)
    genes_text = ('AKT1, AKT2, AKT3, ALK, AR, BRAF, BRCA1, CDKN2A, EGFR, ERBB2, ERBB4, ERG, ESR1, ETV1, ETV4, ETV5, FGFR1, FGFR2, FGFR3, MAP3K8, MET, MTAP, MYB, MYBL1, NOTCH1, NOTCH2, NOTCH3, NRG1, NTRK1, NTRK2, NTRK3, NUTM1, PIK3CA, PIK3CB, PPARG, PRKACA, PRKACB, RAF1, RARA, RELA, RET, ROS1, RSPO2, RSPO3, STAT6, TERT, TFE3, TFEB, YAP1')    
    genes_run = p.add_run(genes_text)
    genes_run.italic = True
    genes_run.font.name = 'Times New Roman'
    genes_run.font.size = Pt(9)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    header_run = p.add_run('Genes Assayed with Full Exon Coverage: ')
    header_run.bold = True
    header_run.underline = True
    header_run.font.name = 'Times New Roman'
    header_run.font.size = Pt(9)
    genes_text = ('ABRAXAS1, ACVR1B, ACVR2A, ADAMTS12, ADAMTS2, AMER1, APC, ARHGAP35, ARID1A, ARID1B, ARID2, ARID5B, ASXL1, ASXL2, ATM, ATR, ATRX, AXIN1, AXIN2, B2M, BAP1, BARD1, BCOR, BLM, BMPR2, BRCA1, BRCA2, BRIP1, CALR, CASP8, CBFB, CD274, CD276, CDC73, CDH1, CDH10, CDK12, CDKN1A, CDKN1B, CDKN2A, CDKN2B, CDKN2C, CHEK1, CHEK2, CIC, CIITA, CREBBP, CSMD3, CTCF, CTLA4, CUL3, CUL4A, CUL4B, CYLD, CYP2C9, CYP2D6, DAXX, DDX3X, DICER1, DNMT3A, DOCK3, DPYD, DSC1, DSC3, ELF3, ENO1, EP300, EPCAM, EPHA2, ERAP1, ERAP2, ERCC2, ERCC4, ERCC5, ERRFI1, ETV6, FANCA, FANCC, FANCD2, FANCE, FANCF, FANCG, FANCI, FANCL, FANCM, FAS, FAT1, FBXW7, FUBP1, GATA3, GNA13, GPS2, HDAC2, HDAC9, HLA-A, HLA-B, HNF1A, ID3, INPP4B, JAK1, JAK2, JAK3, KDM5C, KDM6A, KEAP1, KLHL13, KMT2A, KMT2B, KMT2C, KMT2D, LARP4B, LATS1, LATS2, MAP2K4, MAP2K7, MAP3K1, MAP3K4, MAPK8, MEN1, MGA, MLH1, MLH3, MRE11, MSH2, MSH3, MSH6, MTAP, MTUS2, MUTYH, NBN, NCOR1, NF1, NF2, NOTCH1, NOTCH2, NOTCH3, NOTCH4, PALB2, PARP1, PARP2, PARP3, PARP4, PBRM1, PDCD1, PDCD1LG2, PDIA3, PGD, PHF6, PIK3R1, PMS1, PMS2, POLD1, POLE, POT1, PPM1D, PPP2R2A, PRDM1, PRDM9, PRKAR1A, PSMB10, PSMB8, PSMB9, PTCH1, PTEN, PTPRT, RAD50, RAD51, RAD51B, RAD51C, RAD51D, RAD52, RAD54L, RASA1, RASA2, RB1, RBM10, RECQL4, RNASEH2A, RNASEH2B, RNASEH2C, RNF43, RPA1, RPL22, RPL5, RUNX1, RUNX1T1, SDHA, SDHB, SDHC, SDHD, SETD2, SLX4, SMAD2, SMAD4, SMARCA4, SMARCB1, SOCS1, SOX9, SPEN, STAG2, STAT1, STK11, SUFU, TAP1, TAP2, TBX3, TCF7L2, TET2, TGFBR2, TMEM132D, TNFAIP3, TNFRSF14, TP53, TP63, TPP2, TSC1, TSC2, UGT1A1, USP9X, VHL, WT1, XRCC2, XRCC3, ZBTB20, ZFHX3, ZMYM3, ZRSR2')    
    genes_run = p.add_run(genes_text)
    genes_run.italic = True
    genes_run.font.name = 'Times New Roman'
    genes_run.font.size = Pt(9)
    
    p = document.add_paragraph()
    new_width = Cm(6.85)
    new_height = Cm(0.69)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Methodology.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Nucleic acid (DNA/RNA) was extracted from FFPE tissue samples, using standard Qiagen nucleic acid isolation kits. The DNA/RNA was quantified using Qubit and 20-30 ng of DNA/RNA was amplified using Oncomine Comprehensive assay plus as per the instruction manual. The QC of the library prepared is checked by Agilent Bioanalyser. The 150-200 bp library size samples are sequenced on Ion S5 platform as per manufacturer protocol.')    
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman' 
        
    p = document.add_paragraph()
    new_width = Cm(7.14)
    new_height = Cm(0.69)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Data_analysis.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('The sequence data is processed using the analysis pipeline Ion reporter software 5.18.2.0 as suggested by the vendor. This software can detects and annotates low frequency somatic variants (SNPs, InDels, CNVs) from targeted Ion AmpliSeq™ DNA manual or Ion Chef™ automated libraries, computes automatic tumor cellularity, Loss-of-Heterozygosity, TMB and microsatellite instability (MSI), as well as gene fusions from targeted Ion AmpliSeq™ RNA manual or Ion Chef™ automated libraries, from Oncomine™ Comprehensive Assay Plus run on the Ion 540™ Chip. TMB uses the TMB (Non-Germline Mutations) filter chain with TMB algorithm v4.0. MSI status is computed using a baseline with MSI algorithm v4.0.3. Released with: Ion Reporter™ Software 5.18.4. Workflow Version: 2.4. Samples are classified as TMB-High or TMB-low using a cutoff value of 10mut/mb.')
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman' 
        
    p = document.add_paragraph()
    new_width = Cm(8.23)
    new_height = Cm(0.61)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Interpretation_of_variants.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('The interpretation of the report is based on the clinical information provided. The annotation for variants was derived using various disease databases like dbSNP, ClinVar. The population frequency information from 1000 genomes, ExAC, GnomAD was used for the elimination of common variants/polymorphism. For prediction of the possible impact of coding non-synonymous SNVs on the structure and function of protein, PolyPhen-2, MT2 and SIFT score was used. Further Oncomine Reporter software was used for annotating variants with a curated list of relevant labels, guidelines, and global clinical trials. Oncomine Comprehensive genomic profiling assay will analyze across >500 genes(SNVs,Indels,CNVs,Fusions), plus key immuno-oncology research biomarkers like tumor mutational burden (TMB), microsatellite instability (MSI), and Homologous Recombination Repair genes (HRR).') 
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('The variant is reported as per the Standards and Guidelines for the Interpretation and Reporting of Sequence Variants in Cancer- A Joint Consensus Recommendation of the Association for Molecular Pathology, American Society of Clinical Oncology, and College of American Pathologists.')
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman' 
    p = document.add_paragraph()
    supplementary_table(document)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('The transcript used for clinical reporting generally represents the canonical transcript (according to the Ensembl release 37 gene model), which is usually the longest coding transcript with strong/multiple supporting evidence. However, clinically relevant variants annotated in alternate complete coding transcripts could also be reported. The in silico predictions are based on Variant Effect Predictor, Ensembl release 91 (SIFT version - 5.2.2; PolyPhen - 2.2.2); 2019 release from dbNSFPv4.0 and MutationTaster2 based on build NCBI/ Ensembl 66.')
    for run in p.runs:
        run.font.size = Pt(10)
        run.font.name ='Times New Roman' 

In [30]:
def disclaimer(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(6.00)
    new_height = Cm(0.7)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    image_path = os.path.join("Heading_Images", 'Disclaimer.png')
    run = paragraph.add_run()
    run.add_picture(image_path, width=new_width, height=new_height)
    
    bullet_points = ['This report is based on the sample received in the Lilac Insights laboratory; the analysis is based on the assumption that samples received are representative of the patient demographics mentioned on the test requisition form and the sample tube.',
        'Despite all the necessary precautions and stringency adopted whilst performing DNA tests, the currently available data indicates that the technical error rate associated with all types of DNA analysis, is approximately 2%.',
        'As with all diagnostic tests, the laboratory report must be interpreted in conjunction with the presenting clinical profile of the individual tested and evaluation of all reports. Interpretation of variants is performed based on the current knowledge standards and reporting guidelines. In cases of presence of a VUS, we recommend periodic review of these variants to determine any change in classification based on new published research.',
        'The classification and interpretation of all the variants is carried out based on the current state of scientific knowledge and medical understanding. The results should be correlated clinically.',
        'This report cannot be used for medico legal purposes.']

    for point in bullet_points:
        add_bullet_point(document.add_paragraph(), point)

def add_bullet_point(paragraph, text):
    run = paragraph.add_run(u'\u2022 ') 
    font = run.font
    font.name = 'Times New Roman'
    font.size = Pt(10)
    paragraph.add_run(text)
    paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    paragraph.paragraph_format.left_indent = Cm(2.5)
    paragraph.paragraph_format.right_indent = Cm(2.5)

In [31]:
doc = Document()
add_header_footer(doc)
add_internal_reference_table(doc)
patient_information(doc)
oncoprecise_image(doc)
clinical_history(doc)
report_summary(doc)
therapeutic_summary(doc)
immunotherapy_markers(doc)
approved_immunotherapy(doc)
intrial_immunotherapy(doc)
variant_details(doc)
gene_varient_description(doc)
hrr_genes(doc)
clinical_trial(doc)
supplementary_information(doc)
disclaimer(doc)
doc.save('unipath_report.docx')
#clear_output(wait=True)
message = "<p style='font-size: 24px; font-weight: bold;'>REPORT IS READY FOR DOWNLOAD</p>"
display(Markdown(message))
download_link = f"<a href='unipath_report.docx' download='unipath_report'>Download the Report</a>"
display(HTML(download_link))

No Gene Match
Done with Dna sequence varaints
Done with copy number


<p style='font-size: 24px; font-weight: bold;'>REPORT IS READY FOR DOWNLOAD</p>